# LeetCode 2271 — Maximum White Tiles Covered by a Carpet bằng Python

Link đề: https://leetcode.com/problems/maximum-white-tiles-covered-by-a-carpet/ · Độ khó: **Medium** · Chủ đề: Array, Binary Search, Greedy, Sliding Window, Sorting, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def maximumWhiteTiles(self, tiles: List[List[int]], carpetLen: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Làm việc với dữ liệu dạng **đoạn** (interval) trên trục số rất dài (tới 10⁹) — không thể mở rộng thành từng ô.
2. Chứng minh nhận xét then chốt: **có một vị trí tối ưu mà thảm bắt đầu đúng tại đầu trái của một đoạn trắng** ⇒ chỉ cần thử `n` vị trí.
3. Tính số ô được phủ nhanh bằng **tổng tiền tố độ dài đoạn** + **tìm nhị phân** (O(n log n)), hoặc **cửa sổ trượt hai con trỏ** (O(n)).
4. Xử lý đúng đoạn **bị phủ một phần** ở cuối thảm.
5. Kiểm chứng bằng vét cạn trên tọa độ nhỏ (mở rộng từng ô) với seed cố định.

## 2. Đề bài & ví dụ

`tiles[i] = [l_i, r_i]`: mọi ô `j` với `l_i ≤ j ≤ r_i` là ô **trắng**. Các đoạn **không chồng nhau**. Có một tấm thảm dài `carpetLen` ô, đặt ở đâu cũng được (phủ `carpetLen` ô liên tiếp).

Trả về **số ô trắng lớn nhất** mà thảm phủ được.

**Ví dụ 1:** `tiles = [[1,5],[10,11],[12,18],[20,25],[30,32]], carpetLen = 10` → `9` (đặt thảm từ ô 10 tới 19: phủ `10..11` (2) + `12..18` (7) = 9).

**Ví dụ 2:** `tiles = [[10,11],[1,1]], carpetLen = 2` → `2` (đặt ở 10..11).

**Ràng buộc:** `1 ≤ len(tiles) ≤ 5·10⁴`, `1 ≤ l_i ≤ r_i ≤ 10⁹`, `1 ≤ carpetLen ≤ 10⁹`, các đoạn không chồng nhau (input **có thể chưa sắp xếp**, như ví dụ 2).

In [1]:
from typing import List
from bisect import bisect_right, bisect_left
from itertools import accumulate
import random, time

examples = [([[1, 5], [10, 11], [12, 18], [20, 25], [30, 32]], 10, 9), ([[10, 11], [1, 1]], 2, 2)]
def draw(tiles, start=None, L=None, upto=None):
    upto = upto or max(r for _, r in tiles) + 1
    white = set(j for l, r in tiles for j in range(l, r + 1))
    row = "".join("#" if j in white else "." for j in range(1, upto + 1))
    print("ô  :", row)
    if start is not None:
        print("thảm:", "".join("=" if start <= j < start + L else " " for j in range(1, upto + 1)))
draw(examples[0][0], 10, 10)

ô  : #####....#########.######....###.
thảm:          ==========              


## 3. Kiến thức nền

### 3.1 Sắp xếp đoạn theo đầu trái

Input có thể chưa sắp xếp; `tiles.sort()` (hoặc `sorted`) sắp theo `l` (vì so sánh list theo phần tử đầu trước).

In [2]:
t = [[10, 11], [1, 1], [4, 6]]
print(sorted(t))

[[1, 1], [4, 6], [10, 11]]


### 3.2 Tổng tiền tố độ dài các đoạn

`pre[k]` = tổng độ dài của `k` đoạn đầu (đã sắp). Tổng độ dài các đoạn `i..j−1` = `pre[j] − pre[i]`.

In [3]:
tiles = sorted(examples[0][0])
lens = [r - l + 1 for l, r in tiles]
pre = list(accumulate(lens, initial=0))
print("độ dài:", lens, "| pre:", pre)

độ dài: [5, 2, 7, 6, 3] | pre: [0, 5, 7, 14, 20, 23]


### 3.3 `bisect_right` trên danh sách đầu trái

`bisect_right(starts, x)` = số đoạn có `l ≤ x` ⇒ đoạn cuối cùng **bắt đầu** trước hoặc tại `x` có chỉ số `bisect_right(...) − 1`.

In [4]:
starts = [l for l, _ in tiles]
for x in [9, 10, 19, 31]:
    k = bisect_right(starts, x) - 1
    print(f"x={x}: đoạn cuối có l <= x là chỉ số {k} = {tiles[k]}")

x=9: đoạn cuối có l <= x là chỉ số 0 = [1, 5]
x=10: đoạn cuối có l <= x là chỉ số 1 = [10, 11]
x=19: đoạn cuối có l <= x là chỉ số 2 = [12, 18]
x=31: đoạn cuối có l <= x là chỉ số 4 = [30, 32]


### 3.4 Giao của đoạn thảm với một đoạn trắng

Thảm phủ `[s, e]` với `e = s + carpetLen − 1`. Giao với `[l, r]` có `max(0, min(e, r) − max(s, l) + 1)` ô.

In [5]:
def overlap(s, e, l, r):
    return max(0, min(e, r) - max(s, l) + 1)
print(overlap(10, 19, 12, 18), overlap(10, 19, 20, 25), overlap(10, 19, 1, 5), overlap(15, 24, 12, 18))

7 0 0 4


## 4. Ý tưởng nền tảng / trực giác

**Quan sát then chốt:** giả sử thảm đang ở `[s, e]`. Nếu ô `s` **không** phải là đầu trái của một đoạn trắng thì:
- nếu ô `s` là ô đen (không trắng): dịch thảm sang phải một ô không mất ô trắng nào ở đầu (vì ô bỏ ra là đen), và có thể được thêm một ô ở cuối ⇒ không tệ hơn;
- nếu ô `s` nằm **giữa** một đoạn trắng `[l, r]` (`l < s`): dịch thảm **sang trái** về `l` — phần đầu được thêm các ô trắng `l..s−1` (đúng `s − l` ô), phần cuối mất tối đa `s − l` ô ⇒ không tệ hơn.

Lặp lại ⇒ luôn có lời giải tối ưu với **`s` = đầu trái của một đoạn trắng**. Chỉ cần thử `n` vị trí!

Tính tay ví dụ 1 (`carpetLen = 10`):

| bắt đầu tại | thảm | phủ | số ô trắng |
|---|---|---|---|
| 1 | 1..10 | [1,5] trọn 5 + [10,11] một ô | 6 |
| 10 | 10..19 | [10,11] 2 + [12,18] 7 | **9** |
| 12 | 12..21 | [12,18] 7 + [20,25] 2 | 9 |
| 20 | 20..29 | [20,25] 6 | 6 |
| 30 | 30..39 | [30,32] 3 | 3 |

Max = 9.

In [6]:
tiles = sorted(examples[0][0]); L = 10
for l, _ in tiles:
    s, e = l, l + L - 1
    parts = [(a, b, overlap(s, e, a, b)) for a, b in tiles if overlap(s, e, a, b)]
    print(f"bắt đầu {l:>2}: thảm [{s}, {e}] -> {parts} -> {sum(p[2] for p in parts)}")

bắt đầu  1: thảm [1, 10] -> [(1, 5, 5), (10, 11, 1)] -> 6
bắt đầu 10: thảm [10, 19] -> [(10, 11, 2), (12, 18, 7)] -> 9
bắt đầu 12: thảm [12, 21] -> [(12, 18, 7), (20, 25, 2)] -> 9
bắt đầu 20: thảm [20, 29] -> [(20, 25, 6)] -> 6
bắt đầu 30: thảm [30, 39] -> [(30, 32, 3)] -> 3


## 5. Cách 1 — Brute Force / baseline

### 5.1 Vét cạn trên tọa độ (chỉ khi tọa độ nhỏ)

Mở rộng mọi ô trắng thành tập hợp, thử mọi vị trí bắt đầu từ `min l − carpetLen` tới `max r`.

In [7]:
def brute_coords(tiles, carpetLen):
    white = set(j for l, r in tiles for j in range(l, r + 1))
    lo = min(l for l, _ in tiles); hi = max(r for _, r in tiles)
    best = 0
    for s in range(lo - carpetLen + 1, hi + 1):
        best = max(best, sum(1 for j in range(s, s + carpetLen) if j in white))
    return best

for t, L, exp in examples:
    got = brute_coords(t, L)
    print(got, "OK" if got == exp else "SAI")

9 OK
2 OK


**Độ phức tạp:** O((tọa độ) × carpetLen) — với tọa độ tới 10⁹ là không tưởng.

### 5.2 Vét cạn trên đầu đoạn: thử mỗi `l_i`, cộng giao với mọi đoạn — O(n²)

In [8]:
def brute_starts(tiles, carpetLen):
    best = 0
    for s, _ in tiles:
        e = s + carpetLen - 1
        best = max(best, sum(overlap(s, e, l, r) for l, r in tiles))
    return best

for t, L, exp in examples:
    got = brute_starts(t, L)
    print(got, "OK" if got == exp else "SAI")

9 OK
2 OK


**Độ phức tạp:** O(n²) = 2,5·10⁹ với n = 5·10⁴ ⇒ TLE.

**Nút thắt:** với mỗi vị trí bắt đầu, ta duyệt **mọi** đoạn, trong khi chỉ các đoạn **liên tiếp** sau đoạn bắt đầu (trong phạm vi thảm) mới có giao. Sau khi sắp xếp: các đoạn trọn vẹn trong thảm là một dãy liên tiếp ⇒ tổng tiền tố; đoạn cuối có thể bị phủ một phần ⇒ xử lý riêng.

## 6. Ý tưởng thuật toán / bước đột phá

Sau khi sắp xếp, với thảm bắt đầu tại `l_i` (đầu đoạn `i`), kết thúc tại `e = l_i + carpetLen − 1`:
- Gọi `j` = chỉ số đoạn cuối cùng có `l_j ≤ e` (đoạn cuối mà thảm chạm tới).
- Các đoạn `i..j−1` bị phủ **trọn** (vì đoạn `j` bắt đầu sau chúng và vẫn ≤ e ⇒ chúng kết thúc trước `e`).
- Đoạn `j` bị phủ `min(r_j, e) − l_j + 1` ô (có thể trọn).

`phủ(i) = (pre[j] − pre[i]) + (min(r_j, e) − l_j + 1)`.

**Cách tìm `j`:**
- **Tìm nhị phân** trên mảng đầu trái: `j = bisect_right(starts, e) − 1` ⇒ O(log n) mỗi `i`.
- **Hai con trỏ:** khi `i` tăng, `e` tăng ⇒ `j` không giảm ⇒ duyệt `j` tiến dần, tổng O(n) (cửa sổ trượt).

**Dừng sớm:** nếu đã phủ được `carpetLen` ô (thảm toàn trắng) thì đó là tối đa tuyệt đối.

**Quy tắc "nếu… thì…" của cửa sổ trượt (dạng doocs):** giữ tổng độ dài các đoạn **trọn vẹn** trong thảm `[l_i, e]`; **nếu** đoạn `j` kết thúc `≤ e` **thì** cộng trọn và tiến `j`; **ngược lại** cộng phần `max(0, e − l_j + 1)` của đoạn `j` để cập nhật đáp án; khi `i` tiến, trừ độ dài đoạn `i`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tổng tiền tố + tìm nhị phân

In [9]:
def prefix_binary(tiles, carpetLen):
    t = sorted(tiles); n = len(t)
    starts = [l for l, _ in t]
    pre = list(accumulate((r - l + 1 for l, r in t), initial=0))
    best = 0
    for i in range(n):
        e = t[i][0] + carpetLen - 1
        j = bisect_right(starts, e) - 1            # đoạn cuối có l_j <= e (j >= i)
        cover = pre[j] - pre[i] + min(t[j][1], e) - t[j][0] + 1
        best = max(best, cover)
        if best >= carpetLen: break                # không thể hơn
    return best

for tl, L, exp in examples:
    got = prefix_binary(tl, L)
    print(got, "OK" if got == exp else "SAI")

9 OK
2 OK


**Độ phức tạp cách 2:** O(n log n) (sắp xếp + n lần tìm nhị phân), O(n) bộ nhớ.

### 7.2 Cách 3 — Cửa sổ trượt hai con trỏ (cách doocs)

In [10]:
def sliding_window(tiles, carpetLen):
    t = sorted(tiles); n = len(t)
    best = full = 0; j = 0                          # full = tổng độ dài đoạn trọn vẹn [i, j)
    for i in range(n):
        e = t[i][0] + carpetLen - 1
        while j < n and t[j][1] <= e:              # đoạn j nằm trọn trong thảm
            full += t[j][1] - t[j][0] + 1; j += 1
        partial = max(0, e - t[j][0] + 1) if j < n else 0
        best = max(best, full + partial)
        if best >= carpetLen: break
        full -= t[i][1] - t[i][0] + 1 if j > i else 0
        if j == i: j += 1                          # đoạn i không trọn (thảm ngắn hơn đoạn i)
    return best

for tl, L, exp in examples:
    got = sliding_window(tl, L)
    print(got, "OK" if got == exp else "SAI")

9 OK
2 OK


Chi tiết tinh tế: nếu thảm **ngắn hơn** chính đoạn `i` (`r_i > e`), vòng while không cộng được đoạn nào (`j == i`), phần phủ là `e − l_i + 1 = carpetLen` (tối đa) ⇒ dừng sớm. Dòng `if j == i: j += 1` chỉ để giữ `j ≥ i + 1` cho bước sau, không ảnh hưởng tính đúng.

**Độ phức tạp cách 3:** O(n log n) cho sắp xếp + O(n) cho cửa sổ (mỗi con trỏ đi tối đa `n` bước).

### 7.3 Cách 4 — Thảm **kết thúc** tại đầu phải của một đoạn (đối xứng)

Lập luận đối xứng: cũng có lời giải tối ưu mà thảm **kết thúc** đúng tại `r_j` của một đoạn. Dùng tiền tố + `bisect_left` trên đầu phải.

In [11]:
def ends_at_right(tiles, carpetLen):
    t = sorted(tiles); n = len(t)
    ends = [r for _, r in t]
    pre = list(accumulate((r - l + 1 for l, r in t), initial=0))
    best = 0
    for j in range(n):
        s = t[j][1] - carpetLen + 1
        i = bisect_left(ends, s)                     # đoạn đầu tiên có r_i >= s
        cover = pre[j + 1] - pre[i + 1] + t[i][1] - max(t[i][0], s) + 1
        best = max(best, cover)
    return best

for tl, L, exp in examples:
    got = ends_at_right(tl, L)
    print(got, "OK" if got == exp else "SAI")

9 OK
2 OK


Đoạn `i` (đầu tiên chạm thảm) có thể bị phủ một phần ở **đầu**; các đoạn `i+1..j` phủ trọn.

### 7.4 Cách 5 — Tìm nhị phân trên **tiền tố vị trí** "số ô trắng ≤ x" (hàm đếm)

Định nghĩa `W(x)` = số ô trắng có tọa độ ≤ x (tính bằng tiền tố + bisect). Thảm `[s, e]` phủ `W(e) − W(s − 1)`. Thử mọi `s = l_i`. Cách này rất tổng quát: chỉ cần một hàm đếm tích lũy.

In [12]:
def counting_function(tiles, carpetLen):
    t = sorted(tiles); starts = [l for l, _ in t]
    pre = list(accumulate((r - l + 1 for l, r in t), initial=0))
    def W(x):                                       # số ô trắng có tọa độ <= x
        k = bisect_right(starts, x)                 # các đoạn có l <= x
        if k == 0: return 0
        l, r = t[k - 1]
        return pre[k - 1] + min(r, x) - l + 1
    return max(W(l + carpetLen - 1) - W(l - 1) for l, _ in t)

for tl, L, exp in examples:
    print(counting_function(tl, L), "OK" if counting_function(tl, L) == exp else "SAI")

9 OK
2 OK


## 8. TRACE từng bước

### 8.1 Trace cách 2 trên ví dụ 1

In [13]:
def trace_prefix(tiles, carpetLen):
    t = sorted(tiles); starts = [l for l, _ in t]
    pre = list(accumulate((r - l + 1 for l, r in t), initial=0)); best = 0
    print("đoạn:", t, "| pre:", pre)
    for i in range(len(t)):
        e = t[i][0] + carpetLen - 1; j = bisect_right(starts, e) - 1
        full = pre[j] - pre[i]; part = min(t[j][1], e) - t[j][0] + 1
        best = max(best, full + part)
        print(f"i={i}: thảm [{t[i][0]}, {e}], j={j} ({t[j]}): trọn = {full}, phần cuối = {part} -> {full + part}, best = {best}")
    return best

trace_prefix(*examples[0][:2])

đoạn: [[1, 5], [10, 11], [12, 18], [20, 25], [30, 32]] | pre: [0, 5, 7, 14, 20, 23]
i=0: thảm [1, 10], j=1 ([10, 11]): trọn = 5, phần cuối = 1 -> 6, best = 6
i=1: thảm [10, 19], j=2 ([12, 18]): trọn = 2, phần cuối = 7 -> 9, best = 9
i=2: thảm [12, 21], j=3 ([20, 25]): trọn = 7, phần cuối = 2 -> 9, best = 9
i=3: thảm [20, 29], j=3 ([20, 25]): trọn = 0, phần cuối = 6 -> 6, best = 9
i=4: thảm [30, 39], j=4 ([30, 32]): trọn = 0, phần cuối = 3 -> 3, best = 9


9

### 8.2 Trace cách 3 (cửa sổ trượt) — in `full`, `j` qua từng bước

In [14]:
def trace_window(tiles, carpetLen):
    t = sorted(tiles); n = len(t); best = full = 0; j = 0
    for i in range(n):
        e = t[i][0] + carpetLen - 1; added = []
        while j < n and t[j][1] <= e:
            full += t[j][1] - t[j][0] + 1; added.append(t[j]); j += 1
        partial = max(0, e - t[j][0] + 1) if j < n else 0
        best = max(best, full + partial)
        print(f"i={i}: e={e}, thêm trọn {added}, full={full}, phần đoạn {t[j] if j < n else '-'} = {partial}, best={best}")
        full -= t[i][1] - t[i][0] + 1 if j > i else 0
        if j == i: j += 1
    return best

trace_window(*examples[0][:2])

i=0: e=10, thêm trọn [[1, 5]], full=5, phần đoạn [10, 11] = 1, best=6
i=1: e=19, thêm trọn [[10, 11], [12, 18]], full=9, phần đoạn [20, 25] = 0, best=9
i=2: e=21, thêm trọn [], full=7, phần đoạn [20, 25] = 2, best=9
i=3: e=29, thêm trọn [[20, 25]], full=6, phần đoạn [30, 32] = 0, best=9
i=4: e=39, thêm trọn [[30, 32]], full=3, phần đoạn - = 0, best=9


9

### 8.3 Minh họa lập luận "dịch thảm về đầu đoạn"

Lấy một vị trí bắt đầu bất kỳ `s`, dịch về đầu đoạn theo quy tắc ở mục 4, in số ô phủ trước/sau.

In [15]:
tiles = sorted(examples[0][0]); L = 10
white = set(j for l, r in tiles for j in range(l, r + 1))
def covered(s): return sum(1 for j in range(s, s + L) if j in white)
for s in [3, 7, 14, 22]:
    seg = [(l, r) for l, r in tiles if l <= s <= r]
    if seg: s2 = seg[0][0]; how = f"nằm giữa đoạn {seg[0]} -> dịch trái về {s2}"
    else:
        s2 = min(l for l, _ in tiles if l > s); how = f"ô đen -> dịch phải tới đầu đoạn kế {s2}"
    print(f"s={s:>2}: phủ {covered(s)}; {how}: phủ {covered(s2)}")

s= 3: phủ 6; nằm giữa đoạn (1, 5) -> dịch trái về 1: phủ 6
s= 7: phủ 7; ô đen -> dịch phải tới đầu đoạn kế 10: phủ 9
s=14: phủ 9; nằm giữa đoạn (12, 18) -> dịch trái về 12: phủ 9
s=22: phủ 6; nằm giữa đoạn (20, 25) -> dịch trái về 20: phủ 6


## 9. Code nộp LeetCode

In [16]:
from typing import List
from bisect import bisect_right

class Solution:
    def maximumWhiteTiles(self, tiles: List[List[int]], carpetLen: int) -> int:
        tiles = sorted(tiles)                        # không sửa input
        n = len(tiles)
        starts = [l for l, _ in tiles]
        pre = [0] * (n + 1)                          # pre[k] = tổng độ dài k đoạn đầu
        for k, (l, r) in enumerate(tiles):
            pre[k + 1] = pre[k] + r - l + 1
        best = 0
        for i in range(n):
            e = tiles[i][0] + carpetLen - 1          # thảm bắt đầu ở đầu đoạn i
            j = bisect_right(starts, e) - 1          # đoạn cuối cùng thảm chạm tới
            cover = pre[j] - pre[i] + min(tiles[j][1], e) - tiles[j][0] + 1
            if cover > best:
                best = cover
                if best >= carpetLen:                # thảm toàn trắng: tối đa
                    break
        return best

In [17]:
sol = Solution()
for tl, L, exp in examples:
    got = sol.maximumWhiteTiles(tl, L)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

9 | kỳ vọng 9 | OK
2 | kỳ vọng 2 | OK


## 10. Vì sao thuật toán đúng

**Bổ đề (chuẩn hóa vị trí).** Tồn tại vị trí tối ưu với `s = l_i` cho một `i` nào đó.
*Chứng minh:* lấy vị trí tối ưu `s` sao cho `s` lớn nhất có thể trong các vị trí tối ưu nằm ≤ một đầu đoạn… cách đơn giản hơn: định nghĩa `f(s)` = số ô trắng trong `[s, s+L−1]`. Ta có `f(s+1) − f(s) = [s+L trắng] − [s trắng]`.
- Nếu `s` đen: `f(s+1) ≥ f(s)` ⇒ dịch phải không thiệt; lặp tới khi `s` trắng (hoặc vượt mọi đoạn — khi đó `f = 0`, không tối ưu trừ khi mọi vị trí đều 0, không thể vì `L ≥ 1`).
- Nếu `s` trắng và nằm trong `[l, r]` với `l < s`: `f(s−1) − f(s) = [s−1 trắng] − [s+L−1 trắng] = 1 − [s+L−1 trắng] ≥ 0` ⇒ dịch trái không thiệt; lặp tới `s = l`.
Cả hai quá trình đều dừng, và khi dừng `s` là đầu một đoạn. ∎

**Công thức phủ.** Với `s = l_i`, `e = s + L − 1`, `j = max{k : l_k ≤ e}` (tồn tại vì `l_i ≤ e`, nên `j ≥ i`). Mọi đoạn `k > j` bắt đầu sau `e` ⇒ không giao. Mọi đoạn `k < i` kết thúc trước `l_i` (không chồng nhau, đã sắp) ⇒ không giao. Mọi đoạn `i ≤ k < j` có `r_k < l_j ≤ e` ⇒ phủ trọn. Đoạn `j` phủ `[l_j, min(r_j, e)]`. ⇒ công thức ở mục 6 đúng.

**Dừng sớm.** Số ô phủ ≤ `carpetLen` với mọi vị trí ⇒ khi đạt `carpetLen` không thể tốt hơn.

## 11. Phân tích Time / Space Complexity

`n ≤ 5·10⁴`, tọa độ `X ≤ 10⁹`.

| Cách | Time | Space |
|---|---|---|
| 1a. Vét cạn tọa độ | O(X · carpetLen) | O(số ô trắng) |
| 1b. Thử đầu đoạn, cộng mọi đoạn | O(n²) | O(1) |
| 2. Tiền tố + nhị phân | O(n log n) | O(n) |
| 3. Cửa sổ trượt | O(n log n) (sắp xếp) + O(n) | O(n) |
| 4. Kết thúc ở đầu phải | O(n log n) | O(n) |
| 5. Hàm đếm W(x) | O(n log n) | O(n) |

In [18]:
def rand_tiles(rng, n, gap_max, len_max, start=1):
    out = []; x = start
    for _ in range(n):
        x += rng.randint(1, gap_max); l = x; x += rng.randint(0, len_max); out.append([l, x])
    rng.shuffle(out); return out

rng = random.Random(2271)
big = rand_tiles(rng, 5 * 10**4, 10**4, 10**4)
for L in [1, 10**4, 10**7]:
    res = []
    for name, f in [("prefix_binary", prefix_binary), ("sliding_window", sliding_window), ("ends_at_right", ends_at_right),
                    ("counting_function", counting_function), ("Solution", Solution().maximumWhiteTiles)]:
        t0 = time.perf_counter(); res.append(f(big, L)); dt = time.perf_counter() - t0
        print(f"L={L:<9} {name:<18} -> {res[-1]:<10} {dt*1000:6.0f} ms")
    assert len(set(res)) == 1

L=1         prefix_binary      -> 1              32 ms


L=1         sliding_window     -> 1              26 ms


L=1         ends_at_right      -> 1              42 ms


L=1         counting_function  -> 1              57 ms
L=1         Solution           -> 1              49 ms


L=10000     prefix_binary      -> 10000          39 ms
L=10000     sliding_window     -> 10000          39 ms


L=10000     ends_at_right      -> 10000          48 ms


L=10000     counting_function  -> 10000          61 ms
L=10000     Solution           -> 10000          43 ms


L=10000000  prefix_binary      -> 5223305        50 ms


L=10000000  sliding_window     -> 5223305        47 ms
L=10000000  ends_at_right      -> 5223305        52 ms


L=10000000  counting_function  -> 5223305        64 ms


L=10000000  Solution           -> 5223305        51 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [19]:
ALL = {"brute_coords": brute_coords, "brute_starts": brute_starts, "prefix_binary": prefix_binary,
       "sliding_window": sliding_window, "ends_at_right": ends_at_right, "counting_function": counting_function,
       "Solution": Solution().maximumWhiteTiles}
edge = examples + [
    ([[1, 1]], 1, 1),
    ([[1, 1]], 5, 1),
    ([[1, 10]], 3, 3),                         # thảm ngắn hơn đoạn
    ([[1, 2], [4, 5]], 5, 4),                  # phủ cả hai, ô 3 đen
    ([[1, 2], [4, 5]], 4, 3),                  # 1..4 hoặc 2..5
    ([[8051, 8057], [8074, 8089], [7994, 7995], [7969, 7987], [8013, 8020], [8123, 8139],
      [7930, 7950], [8096, 8104], [7917, 7925], [8027, 8035], [8003, 8011]], 9854, 126),
]
for tl, L, exp in edge:
    for name, f in ALL.items():
        if name == "brute_coords" and (max(r for _, r in tl) - min(l for l, _ in tl) + L) * L > 2 * 10**6: continue
        tt = [list(x) for x in tl]
        assert f(tt, L) == exp, (name, tl, L, f(tl, L), exp)
        assert tt == tl, (name, "đã sửa input")
print("Edge case OK:", len(edge))

Edge case OK: 8


Case cuối (thảm dài hơn mọi đoạn cộng lại): đáp án là **tổng độ dài** mọi đoạn — kiểm tra nhanh: `sum(r − l + 1) = 126`.

In [20]:
tl = edge[-1][0]
print(sum(r - l + 1 for l, r in tl), "| khoảng tọa độ:", max(r for _, r in tl) - min(l for l, _ in tl) + 1, "<= 9854")

126 | khoảng tọa độ: 223 <= 9854


### 12.2 Đối chiếu ngẫu nhiên với vét cạn tọa độ (seed cố định)

In [21]:
rng = random.Random(42)
for _ in range(800):
    tl = rand_tiles(rng, rng.randint(1, 8), rng.choice([1, 3, 10]), rng.choice([0, 2, 6]))
    L = rng.randint(1, 30)
    ref = brute_coords(tl, L)
    for name, f in ALL.items():
        assert f([list(x) for x in tl], L) == ref, (name, tl, L)
print("800 case ngẫu nhiên OK")

800 case ngẫu nhiên OK


### 12.3 Tính chất: đáp án không giảm khi `carpetLen` tăng; ≤ min(carpetLen, tổng độ dài)

In [22]:
rng = random.Random(7)
for _ in range(200):
    tl = rand_tiles(rng, rng.randint(1, 30), 50, 30)
    tot = sum(r - l + 1 for l, r in tl); prev = 0
    for L in sorted(rng.sample(range(1, 500), 10)):
        v = Solution().maximumWhiteTiles(tl, L)
        assert prev <= v <= min(L, tot); prev = v
print("tính chất OK")

tính chất OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không sắp xếp `tiles`

In [23]:
def wrong_no_sort(tiles, carpetLen):
    t = tiles; n = len(t); starts = [l for l, _ in t]                   # SAI: bisect trên danh sách chưa sắp
    pre = list(accumulate((r - l + 1 for l, r in t), initial=0)); best = 0
    for i in range(n):
        e = t[i][0] + carpetLen - 1; j = max(i, bisect_right(starts, e) - 1)
        best = max(best, pre[j] - pre[i] + min(t[j][1], e) - t[j][0] + 1)
    return best

tl = [[10, 11], [1, 1], [3, 8]]
print("bản sai :", wrong_no_sort(tl, 9), "| bản đúng:", Solution().maximumWhiteTiles(tl, 9))

bản sai : 9 | bản đúng: 8


**Sửa:** sắp xếp theo `l` trước (ví dụ 2 của đề đã chưa sắp xếp!).

### Lỗi 2 — Coi đoạn cuối luôn bị phủ trọn

In [24]:
def wrong_full_last(tiles, carpetLen):
    t = sorted(tiles); starts = [l for l, _ in t]
    pre = list(accumulate((r - l + 1 for l, r in t), initial=0)); best = 0
    for i in range(len(t)):
        e = t[i][0] + carpetLen - 1; j = bisect_right(starts, e) - 1
        best = max(best, pre[j + 1] - pre[i])                           # SAI: cộng trọn đoạn j
    return best

print("bản sai :", wrong_full_last(*examples[0][:2]), "| bản đúng:", Solution().maximumWhiteTiles(*examples[0][:2]))

bản sai : 13 | bản đúng: 9


**Vì sao sai:** thảm có thể chỉ phủ một phần đoạn cuối (ví dụ 1: bắt đầu ở 1, thảm 1..10 chỉ chạm ô 10 của `[10, 11]`), thậm chí đáp án vượt `carpetLen`. **Sửa:** `min(r_j, e) − l_j + 1`.

### Lỗi 3 — Lệch một: `e = s + carpetLen`

In [25]:
def wrong_end(tiles, carpetLen):
    t = sorted(tiles); starts = [l for l, _ in t]
    pre = list(accumulate((r - l + 1 for l, r in t), initial=0)); best = 0
    for i in range(len(t)):
        e = t[i][0] + carpetLen                                          # SAI: thảm dài carpetLen + 1
        j = bisect_right(starts, e) - 1
        best = max(best, pre[j] - pre[i] + min(t[j][1], e) - t[j][0] + 1)
    return best

print("bản sai :", wrong_end([[1, 1], [3, 3]], 2), "| bản đúng:", Solution().maximumWhiteTiles([[1, 1], [3, 3]], 2))

bản sai : 2 | bản đúng: 1


**Sửa:** thảm dài `L` phủ `[s, s + L − 1]`.

### Lỗi 4 — Chỉ thử đặt thảm ở **giữa** các đoạn / ở các vị trí "trông đẹp"

In [26]:
def wrong_only_ends(tiles, carpetLen):
    # SAI: chỉ thử thảm KẾT THÚC đúng tại đầu trái của mỗi đoạn (sai hướng chuẩn hóa)
    t = sorted(tiles); white = set(j for l, r in t for j in range(l, r + 1)); best = 0
    for l, _ in t:
        s = l - carpetLen + 1
        best = max(best, sum(1 for j in range(s, l + 1) if j in white))
    return best

print("ví dụ 1 - bản sai :", wrong_only_ends(*examples[0][:2]), "| bản đúng:", Solution().maximumWhiteTiles(*examples[0][:2]), "(trùng hợp)")
print("[[1,1],[5,10]], L=3 - bản sai :", wrong_only_ends([[1, 1], [5, 10]], 3), "| bản đúng:", Solution().maximumWhiteTiles([[1, 1], [5, 10]], 3))

ví dụ 1 - bản sai : 9 | bản đúng: 9 (trùng hợp)
[[1,1],[5,10]], L=3 - bản sai : 1 | bản đúng: 3


**Vì sao sai:** chuẩn hóa đúng là thảm **bắt đầu** tại đầu trái (hoặc **kết thúc** tại đầu phải) của một đoạn. Thảm kết thúc tại **đầu trái** chỉ phủ đúng một ô của đoạn đó — thường rất tệ. **Sửa:** chọn đúng một trong hai chuẩn hóa ở mục 4 / cách 4.

### Lỗi 5 — Mở rộng từng ô thành `set` với tọa độ 10⁹ (tràn bộ nhớ / quá chậm)

In [27]:
def wrong_expand(tiles, carpetLen, limit=2 * 10**6):
    total = sum(r - l + 1 for l, r in tiles)
    if total > limit:
        return f"MemoryError (mô phỏng): cần {total:,} phần tử trong set"
    return brute_coords(tiles, carpetLen)

print(wrong_expand([[1, 10**9]], 5))
print("bản đúng:", Solution().maximumWhiteTiles([[1, 10**9]], 5))

MemoryError (mô phỏng): cần 1,000,000,000 phần tử trong set
bản đúng: 5


**Sửa:** làm việc trên **đoạn** (tiền tố độ dài), không mở rộng ô.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Dữ liệu đoạn trên trục dài ⇒ **sắp xếp + tiền tố độ dài + tìm nhị phân / hai con trỏ**; không mở rộng ô.
- "Đặt một cửa sổ độ dài cố định để phủ nhiều nhất" ⇒ chuẩn hóa vị trí về **mép** của một phần tử dữ liệu (đầu trái hoặc đầu phải), chứng minh bằng "dịch không thiệt".
- Đoạn cuối **bị phủ một phần** — luôn xét riêng.
- Dừng sớm khi đạt cận trên `carpetLen`.
- Bài cùng họ: 1004 Max Consecutive Ones III, 2009, 2054 Two Best Non-Overlapping Events, 2779 Maximum Beauty of an Array After Applying Operation, 1658.

## 15. Biến thể của bài

### 15.1 Biến thể A — **Hai** tấm thảm (không chồng nhau), tối đa số ô trắng

Với dữ liệu nhỏ: dùng hàm đếm `W(x)`; thảm thứ nhất bắt đầu ở đầu đoạn `i`, thảm thứ hai bắt đầu ở đầu đoạn khác **sau** khi thảm thứ nhất kết thúc (hoặc ngay sau đó). Một cách đơn giản và đúng (dữ liệu nhỏ): duyệt cặp vị trí bắt đầu ứng viên gồm các đầu đoạn và "ngay sau thảm thứ nhất".

In [28]:
def two_carpets_brute(tiles, L):
    white = set(j for l, r in tiles for j in range(l, r + 1))
    lo = min(l for l, _ in tiles) - L; hi = max(r for _, r in tiles) + 1
    cov = {s: sum(1 for j in range(s, s + L) if j in white) for s in range(lo, hi + 1)}
    best = 0
    for s1 in cov:
        for s2 in cov:
            if s2 >= s1 + L: best = max(best, cov[s1] + cov[s2])
    return best

def two_carpets(tiles, L):
    t = sorted(tiles); starts = [l for l, _ in t]
    pre = list(accumulate((r - l + 1 for l, r in t), initial=0))
    def W(x):
        k = bisect_right(starts, x)
        if k == 0: return 0
        l, r = t[k - 1]; return pre[k - 1] + min(r, x) - l + 1
    cands = sorted(set([l for l, _ in t] + [l + L for l, _ in t]))
    best = 0
    for s1 in cands:
        for s2 in cands:
            if s2 >= s1 + L:
                best = max(best, W(s1 + L - 1) - W(s1 - 1) + W(s2 + L - 1) - W(s2 - 1))
    # trường hợp chỉ dùng một thảm hữu ích (thảm còn lại đặt ở chỗ trống)
    best = max(best, Solution().maximumWhiteTiles(t, L))
    return best

rng = random.Random(151)
for _ in range(150):
    tl = rand_tiles(rng, rng.randint(1, 6), 4, 4); L = rng.randint(1, 8)
    assert two_carpets(tl, L) == two_carpets_brute(tl, L), (tl, L)
print(two_carpets(*examples[0][:2]), "- hai thảm dài 10 ở ví dụ 1; khớp brute force")

15 - hai thảm dài 10 ở ví dụ 1; khớp brute force


Vì sao chỉ cần các ứng viên "đầu đoạn" và "ngay sau thảm thứ nhất"? Lập luận dịch thảm vẫn áp dụng cho thảm thứ hai, trừ khi nó bị chặn bởi thảm thứ nhất — khi đó vị trí tối ưu của nó là sát ngay sau thảm thứ nhất.

### 15.2 Biến thể B — Mỗi ô trắng có **trọng số** theo đoạn (`w_i` mỗi ô của đoạn `i`)

Thay tiền tố độ dài bằng tiền tố `độ dài × w`; phần đoạn cuối nhân `w_j`. Chuẩn hóa vị trí vẫn đúng? — **không** hoàn toàn: dịch trái về `l` thêm ô trọng số `w_i` ở đầu nhưng có thể mất ô trọng số lớn hơn ở cuối. Cần xét cả hai chuẩn hóa (bắt đầu tại `l_i` **và** kết thúc tại `r_j`). Kiểm chứng:

In [29]:
def weighted_carpet(tiles, w, L):
    order = sorted(range(len(tiles)), key=lambda k: tiles[k][0])
    t = [tiles[k] for k in order]; ww = [w[k] for k in order]
    starts = [l for l, _ in t]; ends = [r for _, r in t]
    pre = list(accumulate(((r - l + 1) * x for (l, r), x in zip(t, ww)), initial=0))
    def Wf(x):
        k = bisect_right(starts, x)
        if k == 0: return 0
        l, r = t[k - 1]; return pre[k - 1] + (min(r, x) - l + 1) * ww[k - 1]
    cands = [l for l, _ in t] + [r - L + 1 for _, r in t]
    return max(Wf(s + L - 1) - Wf(s - 1) for s in cands)

def weighted_brute(tiles, w, L):
    val = {}
    for (l, r), x in zip(tiles, w):
        for j in range(l, r + 1): val[j] = x
    lo = min(l for l, _ in tiles) - L; hi = max(r for _, r in tiles) + 1
    return max(sum(val.get(j, 0) for j in range(s, s + L)) for s in range(lo, hi + 1))

rng = random.Random(152)
for _ in range(300):
    tl = rand_tiles(rng, rng.randint(1, 6), 4, 5); ws = [rng.randint(1, 9) for _ in tl]; L = rng.randint(1, 10)
    assert weighted_carpet(tl, ws, L) == weighted_brute(tl, ws, L)
print("trọng số: xét cả hai chuẩn hóa - khớp brute force")

trọng số: xét cả hai chuẩn hóa - khớp brute force


### 15.3 Biến thể C — Tìm **độ dài thảm nhỏ nhất** để phủ được ít nhất `k` ô trắng

Đáp án đơn điệu theo `carpetLen` (mục 12.3) ⇒ tìm nhị phân trên `carpetLen`, mỗi lần gọi bài gốc. O(n log n · log X).

In [30]:
def min_carpet_for(tiles, k):
    if sum(r - l + 1 for l, r in tiles) < k: return -1
    lo, hi = 1, max(r for _, r in tiles) - min(l for l, _ in tiles) + 1
    while lo < hi:
        mid = (lo + hi) // 2
        if Solution().maximumWhiteTiles(tiles, mid) >= k: hi = mid
        else: lo = mid + 1
    return lo

print(min_carpet_for(examples[0][0], 9), min_carpet_for(examples[0][0], 10), min_carpet_for(examples[0][0], 100))

9 11 -1


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1a. Vét cạn tọa độ | mở rộng ô | O(X·L) | O(ô) | kiểm chứng tọa độ nhỏ |
| 1b. Thử đầu đoạn | cộng mọi đoạn | O(n²) | O(1) | baseline |
| 2. Tiền tố + nhị phân | `pre`, `bisect_right` | O(n log n) | O(n) | **mặc định**, dễ đúng |
| 3. Cửa sổ trượt | hai con trỏ | O(n) sau sắp xếp | O(n) | tối ưu hằng số |
| 4. Kết thúc ở đầu phải | đối xứng | O(n log n) | O(n) | kiểm chứng chéo |
| 5. Hàm đếm W(x) | `W(e) − W(s−1)` | O(n log n) | O(n) | tổng quát (trọng số, nhiều thảm) |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `maximumWhiteTiles([[1,3],[5,8]], 4)` và `maximumWhiteTiles([[1,3],[5,8]], 8)`?

**Bài 2 (tính tay):** Với `tiles = [[2,4],[6,6],[9,15]]`, `carpetLen = 6`: tính số ô phủ khi thảm bắt đầu ở mỗi đầu đoạn.

**Bài 3 (điền TODO):** Viết `bt3_todo(tiles, carpetLen)` trả về **vị trí bắt đầu** của thảm tối ưu (nhỏ nhất nếu hòa trong các đầu đoạn).

**Bài 4 (tự viết lại):** Viết `bt4_todo(tiles, carpetLen)` cài hàm đếm `W(x)` (cách 5) không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(tiles, carpetLen)` trả về số ô **đen** (không trắng) **ít nhất** mà thảm phải phủ, với điều kiện thảm nằm trong `[min l, max r]`.

In [31]:
def bt3_todo(tiles, carpetLen):
    # TODO: vị trí bắt đầu tối ưu
    pass

def bt4_todo(tiles, carpetLen):
    # TODO: hàm đếm W(x)
    pass

def bt5_todo(tiles, carpetLen):
    # TODO: số ô đen ít nhất bị phủ (thảm nằm trong [min l, max r])
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: đoạn dài 3 và 4. Mức 2: thảm 4 ô: bắt đầu ở 5 phủ trọn `[5,8]`. Mức 3: `4`; thảm 8 ô bắt đầu ở 1: phủ `1..8` = 3 + 4 = `7`.

**Bài 2.** Mức 1: bắt đầu 2: thảm `2..7`. Mức 2: phủ `[2,4]` 3 ô + `[6,6]` 1 ô = 4; bắt đầu 6: `6..11` = 1 + 3 = 4. Mức 3: bắt đầu 9: `9..14` = 6 ⇒ đáp án 6.

**Bài 3.** Mức 1: như cách 2 nhưng nhớ `l_i` khi cập nhật. Mức 2: dùng `>` để giữ vị trí nhỏ nhất. Mức 3: trả `t[best_i][0]`.

**Bài 4.** Mức 1: `W(x)` = số ô trắng ≤ x. Mức 2: `k = bisect_right(starts, x)`; nếu `k = 0` trả 0. Mức 3: `pre[k−1] + min(r, x) − l + 1`.

**Bài 5.** Mức 1: số ô đen = `carpetLen − số ô trắng`. Mức 2: thảm nằm trong `[min l, max r]` ⇒ nếu `carpetLen > max r − min l + 1` thì không đặt được (trả −1). Mức 3: tối thiểu ô đen = `carpetLen − (max ô trắng với ràng buộc vị trí)`; ứng viên: các đầu đoạn và các vị trí kết thúc tại đầu phải (để không vượt biên).

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `4`, `7`. **Bài 2:** `4, 4, 6` ⇒ `6`.

In [32]:
assert Solution().maximumWhiteTiles([[1, 3], [5, 8]], 4) == 4
assert Solution().maximumWhiteTiles([[1, 3], [5, 8]], 8) == 7
assert Solution().maximumWhiteTiles([[2, 4], [6, 6], [9, 15]], 6) == 6

def bt3(tiles, carpetLen):
    t = sorted(tiles); starts = [l for l, _ in t]
    pre = list(accumulate((r - l + 1 for l, r in t), initial=0)); best, pos = -1, None
    for i in range(len(t)):
        e = t[i][0] + carpetLen - 1; j = bisect_right(starts, e) - 1
        c = pre[j] - pre[i] + min(t[j][1], e) - t[j][0] + 1
        if c > best: best, pos = c, t[i][0]
    return pos

def bt4(tiles, carpetLen):
    return counting_function(tiles, carpetLen)

def bt5(tiles, carpetLen):
    t = sorted(tiles); lo, hi = t[0][0], t[-1][1]
    if carpetLen > hi - lo + 1: return -1
    starts = [l for l, _ in t]; pre = list(accumulate((r - l + 1 for l, r in t), initial=0))
    def W(x):
        k = bisect_right(starts, x)
        if k == 0: return 0
        l, r = t[k - 1]; return pre[k - 1] + min(r, x) - l + 1
    cands = [s for s in [l for l, _ in t] + [r - carpetLen + 1 for _, r in t] if lo <= s and s + carpetLen - 1 <= hi]
    return carpetLen - max(W(s + carpetLen - 1) - W(s - 1) for s in cands)

def bt5_brute(tiles, carpetLen):
    white = set(j for l, r in tiles for j in range(l, r + 1)); lo = min(l for l, _ in tiles); hi = max(r for _, r in tiles)
    if carpetLen > hi - lo + 1: return -1
    return min(sum(1 for j in range(s, s + carpetLen) if j not in white) for s in range(lo, hi - carpetLen + 2))

rng = random.Random(1971)
for _ in range(500):
    tl = rand_tiles(rng, rng.randint(1, 7), 4, 5); L = rng.randint(1, 25)
    p = bt3(tl, L)
    white = set(j for l, r in tl for j in range(l, r + 1))
    assert sum(1 for j in range(p, p + L) if j in white) == brute_coords(tl, L)
    assert bt4(tl, L) == brute_coords(tl, L)
    assert bt5(tl, L) == bt5_brute(tl, L), (tl, L)
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


Ghi chú bài 5: ràng buộc "thảm nằm trong `[min l, max r]`" có thể ngăn việc đặt thảm bắt đầu ở một đầu đoạn (vì vượt biên phải) — khi đó chuẩn hóa "kết thúc tại đầu phải của một đoạn" cung cấp ứng viên còn thiếu (đặc biệt là đoạn cuối cùng).

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
t = sorted(tiles); starts = [l for l, _ in t]
pre = [0]; [pre.append(pre[-1] + r - l + 1) for l, r in t]
best = 0
for i in range(len(t)):
    e = t[i][0] + carpetLen - 1               # thảm bắt đầu ở đầu đoạn i
    j = bisect_right(starts, e) - 1           # đoạn cuối thảm chạm tới
    best = max(best, pre[j] - pre[i] + min(t[j][1], e) - t[j][0] + 1)
return best
```

**Checklist:**
- [ ] **Sắp xếp** đoạn theo đầu trái.
- [ ] Thảm `[s, s + L − 1]` (lệch một!).
- [ ] Đoạn cuối có thể bị phủ **một phần**.
- [ ] Không mở rộng ô (tọa độ 10⁹).
- [ ] Dừng sớm khi đạt `carpetLen`.